# G01: Benchmark

*Which drug-disease pairs does the grounding experiment ask about, and what does each graph know about them?*

| Item | Definition | Quantification | Scoring |
|---|---|---|---|
| **Positives** | the post-cutoff approval set: approvals after every graph's content date, one row per (drug, disease) | 128 pairs with MONDO and DrugBank ids, regulator, date and evidence link | not scored; this is the benchmark |
| **Per-graph status** | for each KG arm, whether the graph holds both nodes (covered), whether a dossier would carry any evidence for the drug (grounded: at least one gene or pathway neighbour), whether it already carries the edge (leaked), and whether the pair is evaluable (covered, grounded, dated after the graph, not leaked) | counts per graph, and the `common` subset evaluable in every arm | drives the two query families |
| **Candidate pools** | one query per pair: the positive hidden among approved drugs not indicated for the disease or any MONDO descendant of it | pool of 8, seeded, stratified round-robin over therapeutic areas | consumed by G02 |
| **Selection audit** | whether the `common` subset differs from the rest in modality, area, year or regulator count | Fisher odds ratios and Mann-Whitney tests | reported, not scored |

The two families share the same pairs: `main` asks every pair in every arm and stratifies afterwards on the flags each row carries; `common` asks only the pairs evaluable in every KG arm, so the paired comparison between graphs holds evidence availability constant by construction. Covered and grounded are kept apart because a drug can clear the degree floor on drug-drug or side-effect edges alone and still reach the model with no evidence line; the gap between them is a coverage failure the grounding analysis measures rather than hides.

**Reported, not scored.** The selection audit and the distractor-pool statistics are diagnostics on the benchmark; nothing here produces a score.

**Inputs:** `config/params.yaml` (`grounding`) · `data/gold_standards/post_cutoff_approvals/{post_cutoff_approvals.csv, post_cutoff_pair_status.csv}` · `data/gold_standards/bridges/{approved_drug_pool.csv, approved_indications.csv, mondo_hierarchy.csv, do_diseases.csv, mesh_to_doid.csv}`

**Outputs:** `results/{figures,tables}/G01_benchmark/` · `results/checkpoints/G01_benchmark.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

Loads the post-cutoff set and its per-graph status table, then the Open Targets distractor pool. No graph is loaded here; every per-graph fact comes from the status table `tools/build_post_cutoff_approvals.py` wrote.

In [ ]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

from IPython.display import display
import numpy as np
import pandas as pd
import yaml


from biokgsuite.grounding import (build_queries, load_benchmark, load_distractor_pool, mondo_children,
                                  run_bias_tests, selection_frame)
from biokgsuite.viz.grounding import (fig_benchmark_overview, fig_bias_composition, fig_bias_distribution,
                                      fig_bias_forest)
from biokgsuite.io import Outputs, find_config, load_config, require
from biokgsuite.viz.style import setup_style

setup_style()

In [ ]:
# Config, parameters and the results writer.
config = load_config(find_config(_root), trial_subset=False)   # graphs are named below, not by the panel trial file
params = config['analysis_params']
G      = require(params, 'grounding')
BASE   = config['_base_dir']

# Every figure, table and checkpoint goes through `out`, as in notebooks 01-07.
out = Outputs('G01_benchmark', BASE)

KGS       = require(G, 'kgs')
MODELS    = require(G, 'models')
LABELS    = require(G, 'model_labels') | {'none': 'No KG', 'primekg': 'PrimeKG', 'drkg': 'DRKG',
                                           'biokg': 'BioKG', 'openbilink': 'OpenBioLink', 'hetionet': 'Hetionet'}
FAMILIES  = require(G, 'families')
POOL_SIZE = require(G, 'pool_size')
SHUFFLES  = require(G, 'shuffles')
SEEDS     = require(G, 'seeds')
STRICT    = require(G, 'strict')
N_BOOT    = require(G, 'n_bootstrap')
SEED      = require(params, 'random_seed')
RNG       = np.random.default_rng(SEED)
RUN_DIR   = Path(BASE) / 'results' / 'tables' / 'G02_run' / 'runs'

### Loading

In [ ]:
# The benchmark frame: one row per pair with the per-graph flags joined on.
bench = load_benchmark(BASE, kgs=KGS, strict=STRICT)
pool = load_distractor_pool(BASE)
children = mondo_children(BASE)
print(f'{len(bench)} pairs; distractor universe {len(pool[0])} approved drugs; '
      f'{sum(len(v) for v in pool[1].values())} approved indications over {len(pool[1])} diseases')

## 1. The pairs and their per-graph status

In [ ]:
# Coverage, grounding, leakage and evaluability per KG arm, plus the paired subset.
# covered: the graph holds the drug at the degree floor and the disease.
# grounded: the drug has at least one gene or pathway neighbour a dossier renders.
# evaluable: covered, grounded, not leaked, approved after the graph's content date.
status = pd.DataFrame({
    'covered':   {kg: int(bench[f'covered_{kg}'].sum()) for kg in KGS},
    'grounded':  {kg: int(bench[f'grounded_{kg}'].sum()) for kg in KGS},
    'leaked':    {kg: int(bench[f'leaked_{kg}'].sum()) for kg in KGS},
    'evaluable': {kg: int(bench[f'evaluable_{kg}'].sum()) for kg in KGS},
}).rename_axis('kg').reset_index()
status['covered_not_grounded'] = status.covered - status.grounded
status['label'] = status.kg.map(LABELS)
display(status.set_index('label')[['covered', 'grounded', 'covered_not_grounded', 'leaked', 'evaluable']])
n_common = int(bench.common.sum())
print(f"\nevaluable in every arm ('common' family): {n_common} of {len(bench)}"
      f"{'  (strict)' if STRICT else ''}")
print('pairs by number of graphs covering them:', bench.n_kg_covered.value_counts().sort_index().to_dict())
print('pairs by number of graphs with evidence: ', bench.n_kg_grounded.value_counts().sort_index().to_dict())
out.table(status, 'graph_status')
out.table(bench, 'benchmark')


In [ ]:
# Composition of the benchmark by therapeutic area and modality, both of which
# the round-robin sampling and the selection audit use.
comp = (bench.groupby(['area', 'modality']).size().rename('n').reset_index()
        .pivot(index='area', columns='modality', values='n').fillna(0).astype(int))
comp['all'] = comp.sum(axis=1)
display(comp.sort_values('all', ascending=False))
out.table(comp.reset_index(), 'composition')

In [ ]:
# When the approvals fall relative to each graph's content date, what each
# graph can evaluate, and how many graphs can evaluate each pair. Content dates
# come from config/kg_snapshots.yaml, the same file the builder's date check reads.
CONTENT_DATES = {kg: spec.get('content_date')
                 for kg, spec in yaml.safe_load(open(Path(BASE) / 'config' / 'kg_snapshots.yaml'))['snapshots'].items()}
out.figure(fig_benchmark_overview(bench, CONTENT_DATES, KGS), 'G01d_benchmark_overview');


## 2. Candidate pools

One query per pair. The seed fixes both the order of queries (a permutation within each therapeutic area, then round-robin across areas) and the distractor draw, so G02 rebuilds exactly these pools from the seed alone.

In [ ]:
# Pools for the first seed, both families, as a check on what the model will see.
queries = {fam: build_queries(bench, pool, family=fam, pool_size=POOL_SIZE, seed=SEEDS[0], children=children)
           for fam in FAMILIES}
for fam, qs in queries.items():
    print(f'{fam}: {len(qs)} queries, pool size {POOL_SIZE}')
listing = pd.DataFrame([{'family': fam, 'order': i, 'pair_id': q['pair_id'], 'area': q['area'],
                         'disease': q['disease_name'], 'positive': q['drug_name'],
                         'distractors': '; '.join(c['drug_name'] for c in q['candidates'][1:])}
                        for fam, qs in queries.items() for i, q in enumerate(qs)])
display(listing[listing.family == FAMILIES[0]].head(12).drop(columns='family'))
out.table(listing, f'queries_seed{SEEDS[0]}')

In [ ]:
# No distractor may be a correct answer: not indicated for the disease or a
# descendant of it in Open Targets, and not another post-cutoff positive for it.
universe, indicated = pool
violations = 0
for fam, qs in queries.items():
    for q in qs:
        excl = set()
        for m in {q['mondo_id']} | {c for c in children.get(q['mondo_id'], ())}:
            excl |= indicated.get(m, set())
        excl |= set(bench.drugbank_id[bench.mondo_id == q['mondo_id']])
        violations += sum(c['drug_id'] in excl for c in q['candidates'][1:])
assert violations == 0, f'{violations} distractors are indicated for their target disease'
n_pos_in_pool = int(bench.drugbank_id.isin(universe.drug_id).sum())
print(f'no distractor is indicated for its disease; {n_pos_in_pool} of {len(bench)} positives '
      'are themselves in the approved-drug universe')

## 3. Selection audit

Does the `common` family keep a different kind of pair from the ones it drops? If coverage tracks modality or area, the coverage-controlled comparison between graphs is a comparison on a narrower benchmark, and the audit says how much narrower.

In [ ]:
# Kept = evaluable in every KG arm. Traits with fewer than eight pairs are not tested.
sel = selection_frame(bench)
CAT_VARS  = ['Modality', 'Therapeutic area', 'Approval year', 'Regulators', 'First approver']
CONT_VARS = ['approval_year', 'n_agencies']
audit = run_bias_tests(sel, 'kept', CAT_VARS, CONT_VARS, min_n=8)
display(audit.round({'effect': 2, 'ci_low': 2, 'ci_high': 2, 'p': 3}))
out.table(audit, 'selection_audit')

### Visualisation

In [ ]:
TRAITS = {
    'Antibody': sel.Modality == 'antibody',
    'Small molecule': sel.Modality == 'small molecule',
    'Oncology': sel['Therapeutic area'] == 'Oncology',
    'Immunology': sel['Therapeutic area'] == 'Immunology',
    'Two or more regulators': sel.Regulators == 'two or more',
    'Approved 2025 or later': sel.approval_year >= 2025,
}
out.figure(fig_bias_forest(sel, 'kept', TRAITS), 'G01a_selection_forest');
out.figure(fig_bias_composition(sel, 'kept', ['Modality', 'Therapeutic area', 'Regulators']), 'G01b_selection_composition');
out.figure(fig_bias_distribution(sel, 'kept', 'approval_year'), 'G01c_selection_year');

### Checkpoint

In [ ]:
# The benchmark frame and the pool provenance, for the later stages.
prov = (Path(BASE) / 'data' / 'gold_standards' / 'post_cutoff_approvals' / 'PROVENANCE.txt').read_text()
source_sha = next(line.split()[-1] for line in prov.splitlines() if line.startswith('# source_sha256'))
out.checkpoint({
    'benchmark': bench,
    'kgs': KGS, 'strict': STRICT, 'families': FAMILIES, 'pool_size': POOL_SIZE,
    'n_pairs': len(bench), 'n_common': n_common,
    'source_sha256': source_sha,
    'distractor_universe': len(universe),
})
print(f'source table sha256 {source_sha}')

In [ ]:
# Memory cleanup: free state before the next notebook in the pipeline.
for _v in ('kgs', 'graphs', 'dossiers', 'runs', 'bench'):
    globals().pop(_v, None)
gc.collect()
print('done:', out.notebook)